# Temporal averaging of the coastal boundary

MODFLOW~6 integrates a whole coupling interval in one backward-Euler step, and that
step applies the boundary flux across the entire interval: the volume exchanged is
$\Delta t\,C\,(H-h)$. So the boundary head $H$ has to represent the interval, not the
instant at its end. Those coincide only while the boundary varies slowly, and a
tidal boundary sampled over an appreciable fraction of a tidal cycle does not.

Two reductions of the D-Flow FM stage and depth to that single boundary value are
compared, over coarse runs at four coupling intervals scored against a 30-minute
reference:

- **instant** -- the value at the end of the interval. Every scenario before
  August 2026 used this.
- **mean** -- a wetted-fraction weighted time average. For the GHB this is exact
  rather than approximate: the conductance is $C_0$ times a binary wet mask, so the
  interval-mean flux factors as $C_0\langle w\rangle(\langle w s_1\rangle/\langle
  w\rangle - h)$, making the wetted fraction the conductance multiplier and the
  stage a wet-weighted mean.

## Result

The two methods fail in opposite directions, and which one wins is decided by the
tide, not by the model:

- **instant** never damps amplitude, but aliases once the interval exceeds the
  Nyquist limit for the dominant constituent.
- **mean** never aliases, but damps amplitude by an amount that grows with the
  interval.

Below Nyquist the damping penalty exceeds the (absent) aliasing penalty and instant
is closer to the reference; above it, aliasing dominates and the mean wins by a
widening margin. The M2 period is 12.42 h, so the limit is 6.21 h -- and the tracer
error ratio crosses one between the 4 h and 8 h runs, which brackets it.

The rows where instant wins are immaterial in absolute terms: aquifer head RMSE is
under a millimetre at 2 and 4 h coupling for both methods. The interval where the
choice has physical consequence is daily, where instant carries a 33 mm head error
against 3 mm, and inflates peak tracer concentration by 62 %.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy.plot.styles as styles
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
ROOT = pl.Path.cwd().parent
R = ROOT / "results" / "gp"
FIGS = ROOT / "docs" / "GP" / "figures"

SPINUP_D = 5.0          # drop the cold start
M2_HOURS = 12.4206      # principal lunar semidiurnal period
NYQUIST_H = M2_HOURS / 2

# Both 30-minute runs serve as reference. The mean one shares its method with half
# the runs under test, so scoring only against it would flatter them; the instant
# one is the incumbent. A conclusion that depends on which is chosen is not a
# conclusion.
REFS = {"30M instant": "gp_coarse_30.00M_n244",
        "30M mean": "gp_coarse_30.00M_n244_meanbnd"}

# 02/04 use the _instbnd runs, which were verified bit-identical to the
# pre-refactor results on every array.
RUNS = {
    "02.00H": (2 / 24, "gp_coarse_02.00H_n244_instbnd", "gp_coarse_02.00H_n244_meanbnd"),
    "04.00H": (4 / 24, "gp_coarse_04.00H_n244_instbnd", "gp_coarse_04.00H_n244_meanbnd"),
    "08.00H": (8 / 24, "gp_coarse_08.00H_n244", "gp_coarse_08.00H_n244_meanbnd"),
    "01.00D": (1.0, "gp_coarse_01.00D_n244", "gp_coarse_01.00D_n244_meanbnd"),
}
DT_REF = 0.5 / 24
FT2MM = 304.8

In [ ]:
# The statistics and the figure each have one implementation, in
# docs/GP/scripts/. Duplicating them here would let the notebook and the manuscript
# drift apart, which is how the sequence figure once came to disagree with the text
# describing it.
sys.path.insert(0, str(ROOT / "docs" / "GP" / "scripts"))
import boundary_averaging_data as bad

ds, source = bad.load_or_refresh()
print("recomputed from results/" if source == "results"
      else f"read archive; {len(bad.missing())} runs absent")
stats = ds.to_dataframe().reset_index()
stats["hours"] = stats["interval"].map(dict(zip(ds.interval.values, ds.hours.values)))
PEAK_REF = float(ds.attrs["peak_reference_concentration"])
NYQUIST_H = float(ds.attrs["m2_nyquist_hours"])
print(f"Nyquist limit for M2: {NYQUIST_H:.2f} h;  reference peak tracer {PEAK_REF:.4f}")

In [ ]:
for rlabel in REFS:
    s = stats[stats["ref"] == rlabel].set_index("interval")
    print(f"\n=== reference: {rlabel} ===")
    print("ratio = instant RMSE / mean RMSE;  > 1 means the mean is closer\n")
    print(s[["hours", "head_inst", "head_mean", "head_ratio",
             "seep_inst", "seep_mean", "seep_ratio",
             "trac_inst", "trac_mean", "trac_ratio"]]
          .rename(columns={"head_inst": "head_i(mm)", "head_mean": "head_m(mm)"})
          .round(4).to_string())

s = stats[stats["ref"] == "30M instant"].set_index("interval")
print("\n\npeak sewer tracer concentration, reference %.4f" % PEAK_REF)
print((100 * (s[["peak_inst", "peak_mean"]] - PEAK_REF) / PEAK_REF)
      .rename(columns={"peak_inst": "instant %err", "peak_mean": "mean %err"})
      .round(1).to_string())

In [ ]:
# The figure itself is drawn by docs/GP/scripts/make_boundary_averaging_figure.py,
# from this summary rather than from the simulation output. The output is tens of
# gigabytes and is not in version control, so a co-author with only the repository
# could not otherwise rebuild a manuscript figure. Keeping one plotting
# implementation also stops the notebook and the manuscript drifting apart.
ds = stats.set_index(["ref", "interval"]).to_xarray()
_hours = ds["hours"].isel(ref=0).values
ds = ds.drop_vars("hours").assign_coords(hours=("interval", _hours))
ds["hours"].attrs = {"units": "hours", "long_name": "coupling interval"}
for v, u in [("head_inst", "mm"), ("head_mean", "mm"),
             ("seep_inst", "ft3/d"), ("seep_mean", "ft3/d"),
             ("trac_inst", "1"), ("trac_mean", "1"),
             ("peak_inst", "1"), ("peak_mean", "1")]:
    ds[v].attrs["units"] = u
ds.attrs = {
    "title": "Coupled-solution error against a 30-minute reference, by boundary "
             "reduction and coupling interval",
    "summary": "RMSE of the coarse-grid coupled solution relative to a 30-minute "
               "reference simulation, for an end-of-interval (inst) and a "
               "wetted-fraction time-averaged (mean) reduction of the D-Flow FM "
               "coastal boundary. Both a sampled and an averaged reference are "
               "scored; they agree.",
    "source": "notebooks-GP/step3_plot_compare_boundary_averaging.ipynb",
    "spinup_days_excluded": SPINUP_D,
    "m2_period_hours": M2_HOURS,
    "m2_nyquist_hours": NYQUIST_H,
    "peak_reference_concentration": PEAK_REF,
    "grid": "coarse",
}
NC = ROOT / "docs" / "data" / "boundary_averaging.nc"
NC.parent.mkdir(parents=True, exist_ok=True)
ds.to_netcdf(NC)
print("wrote", NC, NC.stat().st_size, "bytes")

import runpy
runpy.run_path(str(ROOT / "docs" / "GP" / "scripts"
                   / "make_boundary_averaging_figure.py"), run_name="__main__")

In [ ]:
# Kept for interactive use; the manuscript figure is the one written above.
C_I, C_M = "#d62728", "#1f77b4"
s = stats[stats["ref"] == "30M instant"].sort_values("hours")
h = s["hours"].to_numpy()

with styles.USGSPlot():
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(7.5, 5.2), layout="constrained")
    panels = [(axs[0, 0], "head_inst", "head_mean", "Aquifer head RMSE, in millimeters", True),
              (axs[0, 1], "seep_inst", "seep_mean", "Sewer seepage RMSE, in cubic feet per day", True),
              (axs[1, 0], "trac_inst", "trac_mean", "Sewer tracer RMSE, dimensionless", True)]
    for i, (ax, ci, cm, lab, logy) in enumerate(panels):
        ax.plot(h, s[ci], "o-", color=C_I, lw=1.2, ms=4, label="instantaneous")
        ax.plot(h, s[cm], "s-", color=C_M, lw=1.2, ms=4, label="time-averaged")
        ax.set_xscale("log")
        if logy:
            ax.set_yscale("log")
        ax.axvline(NYQUIST_H, color="0.35", lw=0.9, linestyle=(0, (3, 2)), zorder=1)
        ax.set_xticks([2, 4, 8, 24])
        ax.set_xticklabels(["2 h", "4 h", "8 h", "1 d"], fontsize=7)
        ax.tick_params(labelsize=7, top=False)
        styles.heading(ax=ax, letter="ABCD"[i], heading=lab, fontsize=7.5)

    ax = axs[1, 1]
    ax.axhline(PEAK_REF, color="0.35", lw=0.9, linestyle=(0, (3, 2)), zorder=1)
    ax.plot(h, s["peak_inst"], "o-", color=C_I, lw=1.2, ms=4)
    ax.plot(h, s["peak_mean"], "s-", color=C_M, lw=1.2, ms=4)
    ax.axvline(NYQUIST_H, color="0.35", lw=0.9, linestyle=(0, (3, 2)), zorder=1)
    ax.set_xscale("log")
    ax.set_xticks([2, 4, 8, 24])
    ax.set_xticklabels(["2 h", "4 h", "8 h", "1 d"], fontsize=7)
    ax.tick_params(labelsize=7, top=False)
    styles.heading(ax=ax, letter="D", heading="Peak sewer tracer concentration",
                   fontsize=7.5)
    ax.annotate("reference", xy=(2.05, PEAK_REF), xytext=(2.05, PEAK_REF * 1.04),
                fontsize=6.5, color="0.35")

    for ax in axs.flat:
        styles.xlabel(ax=ax, label="Coupling interval")
        ax.annotate(r"$M_2$ Nyquist", xy=(NYQUIST_H, ax.get_ylim()[1]),
                    xytext=(NYQUIST_H * 1.06, ax.get_ylim()[1]),
                    fontsize=6.5, color="0.35", va="top")
    hs, ls = axs[0, 0].get_legend_handles_labels()
    styles.graph_legend(ax=axs[1, 0], handles=hs, labels=ls, loc="lower center",
                        bbox_to_anchor=(1.05, -0.42), ncol=2, frameon=False, fontsize=7.5)

### Reading the figure

Panels A--C are RMSE against the 30-minute reference; D is the peak tracer
concentration, with the reference value dashed. The vertical rule is the Nyquist
limit for M2, 6.21 h.

Below the limit the two curves are close and both are small -- aquifer head RMSE is
under a millimetre at 2 and 4 h -- so the choice there is immaterial even where the
ratio favours instantaneous sampling. Above it the curves separate: instantaneous
peak tracer concentration leaves the reference abruptly, reaching +27 % at 8 h and
+62 % at daily coupling, while the averaged boundary stays within a couple of
percent until daily.

That abruptness is the signature. A first-order truncation error grows smoothly with
the step; aliasing does not appear at all until the sampling interval crosses the
limit, and then grows quickly. The two methods are not better and worse versions of
the same approximation -- they fail by different mechanisms, and the tide decides
which mechanism is active.